# Air quality
## Mexicali Urban Liveability Index — `WP03_air_quality`

**Lead:** TBC
**Indicators assigned:** 6
**Schema version:** 1.0.0

Ambient air quality from the Mexicali monitoring network and satellite/reanalysis products: pollutant concentrations, exceedance days against WHO guidance, and an overall air quality assessment.

These rows are one construct seen through different lenses and time bases -- an index value, that value evaluated against a standard, and the frequency of compliance across a year -- so deliver them as a single measure family sharing a method and data sources, distinguished by the `temporal_basis` and `threshold` fields. The monitoring network has few stations, so declare the native scale and interpolation method honestly; do not present station values as if they were 100 m resolution.

> New to this project? Work through
> [`00_overview_and_schema.ipynb`](00_overview_and_schema.ipynb)
> first — it carries one indicator end to end. Then read
> [`docs/analyst_guide.md`](../docs/analyst_guide.md).

## How to work through this notebook

For each indicator assigned to you, in this order:

1. **Read the brief.** It reproduces everything the team already
   recorded in the workbook — the draft rationale, the article the
   indicator was adapted from, candidate data sources, and the open
   questions colleagues raised. Do not retype any of it; it is
   already in your metadata stub.
2. **Write the causal pathway sentence** (guide §2.1) and find
   **independent health evidence** for it (§2.2). Do this *before*
   looking for data. Fill in `meta['rationale']`.
3. **Find and document the data** (§3): citation, URL, date
   retrieved, licence, and whether it reaches Condesa.
4. **Compute** at the finest scale your data genuinely support.
   Produce a `DataFrame` with `geo_id` and `value`.
5. **Harmonise** with `uli.harmonise(...)`, label with
   `uli.label(...)`, and **deliver** with
   `uli.write_indicator(...)`.
6. **Look at the map.** Most errors are obvious in ten seconds and
   invisible in a table.

`uli.write_indicator` validates first and refuses to publish a
failing deliverable. While you are still iterating, pass
`allow_failure=True` to write a draft anyway.

Full guidance: [`docs/analyst_guide.md`](../docs/analyst_guide.md).
Schema: [`schema/ULI_output_schema.md`](../schema/ULI_output_schema.md).

## Framing the indicator against health evidence

Every indicator must be justified by evidence of a **meaningful
health or wellbeing benefit**, independent of the liveability
article it was adapted from. Those articles establish that an
indicator is used; they rarely establish that it matters.

Complete this sentence before you compute anything:

> *[what I measure]* changes *[a mechanism]*, which changes *[a
> behaviour or exposure]*, which affects *[a health outcome]*.

For most indicators in this project the behaviour is **walking for
transport**, **walking or recreation in public space**, or
**social contact** — and the exposure is **heat**, **air
pollution** or **injury risk**. Say which, using the vocabulary in
`uli.vocab.HEALTH_PATHWAYS`.

Prefer meta-analyses and systematic reviews, then reputable
guidance (WHO, UN-Habitat, PAHO, Secretaría de Salud), then cohort
studies and natural experiments. Record the **effect size with its
uncertainty**.

**If the evidence supports a different threshold from the one the
workbook proposes, use the evidence-based threshold** and say so in
`threshold_justification`. That is explicitly what the project
wants.

**Mexicali is arid and extremely hot.** Most of this literature
comes from temperate cities. Where the transfer is doubtful — for
example, distance-based walkability thresholds in a city where
summer maxima exceed 45 °C and shade rather than distance is the
binding constraint — record it in `rationale.arid_context`. That is
a contribution, not a caveat.

## When several workbook rows are really one indicator

The workbook harvested indicators article by article, so a single
construct sometimes appears as several rows seen through different
lenses or over different time periods. Air quality is the clearest
case:

| Row | What it is | Lens | Time basis |
|---|---|---|---|
| #292 Air quality | the index value itself | `quality` | `annual_mean` |
| #8 Good air quality | that value against a standard | `quality` | `threshold_share` |
| #293 Days with good air quality | how often the standard is met | `quantity` | `threshold_compliance_days` |
| #173 Days PM2.5 over WHO | the same, for one pollutant | `quantity` | `threshold_exceedance_days` |

These are not four indicators — they are one construct measured
four ways, and computing them separately would mean four
inconsistent methods and four sets of data documentation.

Deliver them as a **measure family**: give every measure the same
`measure_family` slug, and distinguish them with `temporal_basis`
(see `uli.vocab.TEMPORAL_BASES`) and `threshold`. They can still
live under separate workbook ids — the family slug is what tells
the index step, and Reimagina Urbana, that they belong together.

```python
for meta in (meta_292, meta_8, meta_293, meta_173):
    for measure in meta['measures']:
        measure['measure_family'] = 'air_quality'
    meta['data_sources'] = SHARED_SOURCES   # one method, one source
```

The same pattern applies to mean summer temperature versus days
above a comfort threshold (WP02), and to flood extent versus annual
average days of flooding (WP05).

## Condesa coverage is a requirement, not a nicety

The Condesa new development in south-east Mexicali is a project
focus area, and it defeats the usual assumptions:

- about **20%** of it falls outside the previously configured
  study region boundary;
- only **44%** of its area is covered by census manzana polygons,
  so a **manzana-native calculation reaches 33 of the 40
  fraccionamientos, while a `grid_100m`-native one reaches all
  40**;
- it is platted and roaded (43 km of street network in OpenStreetMap
  across 27 of the 40 fraccionamientos) but essentially unbuilt —
  **zero destinations**, and satellite-derived population products
  see almost nobody there.

**One thing is your decision: the native scale.** If your data
allow it, compute on the 100 m grid. That is the difference between
reaching all of Condesa and quietly missing a fifth of it.

Everything else is handled downstream. Population denominators,
the 2030 occupancy scenario and population-weighted exposure
statistics are a reporting-step concern (`uli.exposure`), decided
once for the whole project rather than by each analyst. Urban
fabric and exposure measures — land cover, air quality, heat,
hazards, street infrastructure — are properties of *place*, and
should be computed as such; who lives there is applied later.

Two things to record, though:

- `data_sources[].condesa_coverage` — whether your **source**
  reaches Condesa. Satellite imagery and OSM generally do; a 2020
  census variable or a household survey generally does not.
- `method.condesa_treatment` — what you did about it. Where a
  source does not reach Condesa, mark those rows `no_data` rather
  than omitting them.

The validator treats poor Condesa coverage as an **error**.

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

        WORK_PACKAGE = 'WP03_air_quality'
        NOTEBOOK = 'notebooks/03_air_quality.ipynb'

---
## Your indicators (6)

Each has a brief reproducing what the workbook records,
then three working cells: documentation, calculation,
delivery.

### 8 — Good air quality

`good_air_quality` · *Ambient Environment · Risks · Air Quality · Good air quality*

- **Lenses to deliver:** quality
- **Draft rationale (rewrite this):** Suitable air quality is a fundamental environmental requirement for reducing risks of respiratory and cardiovascular diseases in urban populations.
- **Adapted from:** #16: Kashi (2025), ‘Spatial analysis and ranking of urban districts based on a comprehensive livability approach: the case of Tehran’
- **Candidate data sources:** Official Air Quality monitoring system: https://www.mexicali.gob.mx/portalmexicali/calidad-aire/indices?id=1

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_8) at any time to list what is
# still outstanding.
meta_8 = uli.metadata_stub(8, analyst=ANALYST)

# meta_8['rationale']['statement'] = """..."""
# meta_8['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_8['rationale']['arid_context'] = '...'
# meta_8['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_8['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_8)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_8 = 'grid_100m'
METHOD_8 = 'population_weighted_mean'

native_8 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_8 = uli.harmonise(
    native_8,
    native_scale=NATIVE_SCALE_8,
    method=METHOD_8,
)
results_8 = uli.label(
    harmonised_8,
    meta_8,
    measure_id='good_air_quality__quality',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_8, meta_8))
# uli.write_indicator(results_8, meta_8)

### 136 — Annual average nitrogen dioxide (1-e6 mmol/m²)

`annual_average_nitrogen_dioxide_1_e6_mmol_m2` · *Ambient Environment · Risks · Air Quality · Annual average nitrogen dioxide (1-e6 mmol/m²)*

- **Lenses to deliver:** quantity, quality
- **Draft rationale (rewrite this):** Nitrogen dioxide is identified as a major urban problem impacting health and wellbeing, with high concentrations linked to respiratory and cardiovascular issues.
- **Adapted from:** #13: Alderton (2021), ‘Measuring and monitoring liveability in a low-to-middle income country: a proof-of-concept for Bangkok, Thailand and lessons from an international partnership’
- **Candidate data sources:** Sentinel-5P NRTI NO2: Near Real-Time Nitrogen Dioxide (https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_NRTI_L3_NO2?hl=es-419)
- **Feasibility flag:** yes, data believed available

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_136) at any time to list what is
# still outstanding.
meta_136 = uli.metadata_stub(136, analyst=ANALYST)

# meta_136['rationale']['statement'] = """..."""
# meta_136['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_136['rationale']['arid_context'] = '...'
# meta_136['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_136['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_136)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_136 = 'grid_100m'
METHOD_136 = 'population_weighted_mean'

native_136 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_136 = uli.harmonise(
    native_136,
    native_scale=NATIVE_SCALE_136,
    method=METHOD_136,
)
results_136 = uli.label(
    harmonised_136,
    meta_136,
    measure_id='annual_average_nitrogen_dioxide_1_e6_mmol_m2__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_136, meta_136))
# uli.write_indicator(results_136, meta_136)

### 173 — Number of days PM 2.5 exceeds WHO standard (25 µg/m³)

`number_of_days_pm_2_5_exceeds_who` · *Ambient Environment · Risks · Air Quality · Number of days PM 2.5 exceeds WHO standard (25 µg/m³)*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Monitoring PM2.5 levels relative to international WHO standards ensures that the city is tracking environmental risks that impact population cardiovascular and respiratory health.
- **Adapted from:** #13: Alderton (2021), ‘Measuring and monitoring liveability in a low-to-middle income country: a proof-of-concept for Bangkok, Thailand and lessons from an international partnership’
- **Candidate data sources:** Official Air Quality monitoring system: https://www.mexicali.gob.mx/portalmexicali/calidad-aire/indices?id=1 Review historial data from this source

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_173) at any time to list what is
# still outstanding.
meta_173 = uli.metadata_stub(173, analyst=ANALYST)

# meta_173['rationale']['statement'] = """..."""
# meta_173['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_173['rationale']['arid_context'] = '...'
# meta_173['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_173['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_173)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_173 = 'grid_100m'
METHOD_173 = 'population_weighted_mean'

native_173 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_173 = uli.harmonise(
    native_173,
    native_scale=NATIVE_SCALE_173,
    method=METHOD_173,
)
results_173 = uli.label(
    harmonised_173,
    meta_173,
    measure_id='number_of_days_pm_2_5_exceeds_who__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_173, meta_173))
# uli.write_indicator(results_173, meta_173)

### 252 — Per capita SO2 emissions

`per_capita_so2_emissions` · *Ambient Environment · Risks · Air Quality · Per capita SO2 emissions*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Lower levels of sulfur dioxide emissions signify reduced industrial pollution and progress toward carbon reduction, which are necessary for a sustainable living environment.
- **Adapted from:** #31: Xiao (2022), ‘Assessing spatial–temporal evolution and key factors of urban livability in arid zone: The case study of the Loess Plateau, China’; #3: Xiao (2022), ‘Assessment and key factors of urban liveability in underdeveloped regions: A case study of the Loess Plateau, China’
- **Team notes:** ER: Review if there is available data from SEMARNAT for: Bióxido de azufre (SO2) "We could potentially use data from: Registro de Emisiones y Transferencias de Contaminantes (RETC) 2004-2024 (https://sinat.semarnat.gob.mx:8443/retc/retc/index.php) or Taking Stock on industry (https://takingstock.cec.org/Map?IndustryLevel=4&Measure=3&MediaTypes=2&ReportType=1&ResultType=1&Years=2023)"

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_252) at any time to list what is
# still outstanding.
meta_252 = uli.metadata_stub(252, analyst=ANALYST)

# meta_252['rationale']['statement'] = """..."""
# meta_252['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_252['rationale']['arid_context'] = '...'
# meta_252['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_252['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_252)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_252 = 'grid_100m'
METHOD_252 = 'population_weighted_mean'

native_252 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_252 = uli.harmonise(
    native_252,
    native_scale=NATIVE_SCALE_252,
    method=METHOD_252,
)
results_252 = uli.label(
    harmonised_252,
    meta_252,
    measure_id='per_capita_so2_emissions__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_252, meta_252))
# uli.write_indicator(results_252, meta_252)

### 292 — Air quality

`air_quality` · *Ambient Environment · Risks · Air Quality · Air quality*

- **Lenses to deliver:** quality
- **Draft rationale (rewrite this):** Better air quality prioritizes physical health and promotes overall well-being by reducing exposure to harmful atmospheric pollutants.
- **Adapted from:** #18: Jodder (2024), ‘Urban Livability in a Rapidly Urbanizing Mid-Size City: Lessons for Planning in the Global South’; #22: Alderton (2019), ‘What is the meaning of urban liveability for a city in a low-to-middle-income country? Contextualising liveability for Bangkok, Thailand’
- **Candidate data sources:** Official Air Quality monitoring system: https://www.mexicali.gob.mx/portalmexicali/calidad-aire/indices?id=1
- **Open questions raised:** ER: Too broad (should be defined by particles)?
- **Feasibility flag:** to be determined

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_292) at any time to list what is
# still outstanding.
meta_292 = uli.metadata_stub(292, analyst=ANALYST)

# meta_292['rationale']['statement'] = """..."""
# meta_292['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_292['rationale']['arid_context'] = '...'
# meta_292['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_292['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_292)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_292 = 'grid_100m'
METHOD_292 = 'population_weighted_mean'

native_292 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_292 = uli.harmonise(
    native_292,
    native_scale=NATIVE_SCALE_292,
    method=METHOD_292,
)
results_292 = uli.label(
    harmonised_292,
    meta_292,
    measure_id='air_quality__quality',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_292, meta_292))
# uli.write_indicator(results_292, meta_292)

### 293 — Number of days with good air quality

`number_of_days_with_good_air_quality` · *Ambient Environment · Risks · Air Quality · Number of days with good air quality*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** The frequency of days with high-quality air is a fundamental indicator of urban environmental health and atmospheric safety for residents.
- **Adapted from:** #3: Xiao (2022), ‘Assessment and key factors of urban liveability in underdeveloped regions: A case study of the Loess Plateau, China’; #31: Xiao (2022), ‘Assessing spatial–temporal evolution and key factors of urban livability in arid zone: The case study of the Loess Plateau, China’
- **Candidate data sources:** Official Air Quality monitoring system: https://www.mexicali.gob.mx/portalmexicali/calidad-aire/indices?id=1 Review historial data from this source

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_293) at any time to list what is
# still outstanding.
meta_293 = uli.metadata_stub(293, analyst=ANALYST)

# meta_293['rationale']['statement'] = """..."""
# meta_293['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_293['rationale']['arid_context'] = '...'
# meta_293['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_293['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_293)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_293 = 'grid_100m'
METHOD_293 = 'population_weighted_mean'

native_293 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_293 = uli.harmonise(
    native_293,
    native_scale=NATIVE_SCALE_293,
    method=METHOD_293,
)
results_293 = uli.label(
    harmonised_293,
    meta_293,
    measure_id='number_of_days_with_good_air_quality__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_293, meta_293))
# uli.write_indicator(results_293, meta_293)

---
## Check what this work package has delivered

In [ ]:
delivered, catalogue = uli.collect()
if len(catalogue):
    display(catalogue)
    print(delivered.groupby(['indicator_code', 'geo_level']).size())
else:
    print('Nothing delivered yet.')

In [ ]:
# Sanity-check a delivered measure on a map before you call it done.
# MEASURE = 'your_indicator_code__quantity'
# LEVEL = 'manzana'
# units = uli.geography.load(LEVEL).merge(
#     delivered.query('measure_id == @MEASURE and geo_level == @LEVEL'),
#     on='geo_id', how='left')
# ax = units.plot(column='value', legend=True, figsize=(11, 8),
#                 missing_kwds={'color': 'lightgrey'})
# condesa = uli.geography.load('condesa_fraccionamiento')
# condesa.boundary.plot(ax=ax, color='red', linewidth=1)
# ax.set_title(MEASURE)
# ax.set_axis_off()